# Sphere Octant Division With Area Optimizer (`N^2`)

Divide the first sphere octant (`x, y, z >= 0`) into `N²` spherical triangles and visualize them.

- The mesh is built from a simplex lattice and normalized onto the sphere.
- Visualization uses `matplotlib` 3D plotting.
- The last cell checks side-length patterns.


In [ ]:
import numpy as np

from pathlib import Path

from coordinate_fileio import save_division_result
from sphere_division_algorithms import (
    build_point_index,
    build_projected_positions,
    positions_permutation_error,
    run_tension_equalizer,
    spherical_triangle_areas,
)
from sphere_division_visualization import (
    plot_before_after_mesh_comparison,
    plot_optimizer_history_and_distribution,
)


## Deterministic tension iterator for equal spherical areas

Start from the existing octant mesh and iteratively move vertices so spherical triangle areas approach the global mean.

### Optimization algorithm

Let the spherical triangles be denoted by $T_t = (\mathbf{a}_t, \mathbf{b}_t, \mathbf{c}_t)$, and let their spherical areas be

$$A_t = A(\mathbf{a}_t, \mathbf{b}_t, \mathbf{c}_t)$$

In the implementation, the area is evaluated by

$$A(\mathbf{a}, \mathbf{b}, \mathbf{c}) = 2 \arctan \left( \frac{|\mathbf{a} \cdot (\mathbf{b} \times \mathbf{c})|}{1 + \mathbf{a}\cdot\mathbf{b} + \mathbf{b}\cdot\mathbf{c} + \mathbf{c}\cdot\mathbf{a}} \right)$$

where every vertex stays on the unit sphere inside the first octant.

At iteration $k$, the global mean area is

$$\bar{A}^{(k)} = \frac{1}{|\mathcal{T}|} \sum_{t \in \mathcal{T}} A_t^{(k)}$$

and the signed relative area error of triangle $t$ is

$$r_t^{(k)} = \frac{A_t^{(k)} - \bar{A}^{(k)}}{\max(\bar{A}^{(k)}, \varepsilon)}$$

The center direction used by the optimizer is the normalized centroid

$$\mathbf{g}_t^{(k)} = \frac{\mathbf{a}_t^{(k)} + \mathbf{b}_t^{(k)} + \mathbf{c}_t^{(k)}}{\|\mathbf{a}_t^{(k)} + \mathbf{b}_t^{(k)} + \mathbf{c}_t^{(k)}\|}$$

For a vertex $\mathbf{v}$ adjacent to triangle $t$, the triangle proposes

$$\Delta_{t \to v}^{(k)} = r_t^{(k)} \bigl( \Pi_v(\mathbf{g}_t^{(k)}) - \mathbf{v}^{(k)} \bigr)$$

where $\Pi_v$ is the projection onto the feasible set of that vertex.

All proposals received by the same vertex are averaged:

$$\bar{\Delta}_v^{(k)} = \frac{1}{d(v)} \sum_{t: v \in T_t} \Delta_{t \to v}^{(k)}$$

where $d(v)$ is the number of incident triangles.

The actual update is

$$\mathbf{v}^{(k+1)} = \Pi_v \bigl( \mathbf{v}^{(k)} + \eta^{(k)} \bar{\Delta}_v^{(k)} \bigr)$$

Here $\eta^{(k)}$ is the learning rate. When both the area standard deviation and the maximum relative deviation stop improving, the code applies a mild decay:

$$\eta^{(k+1)} = 0.99 \, \eta^{(k)}$$

So the method is a deterministic projected fixed-point iteration that reduces spherical-area variance while preserving octant and boundary constraints.


In [ ]:
# Run iterator
N = 16
positions_eq, triangle_keys_eq, hist = run_tension_equalizer(
    N, iterations=1000, lr=0.2, verbose_every=50
)

# Final spherical area distribution
areas_eq = spherical_triangle_areas(positions_eq, triangle_keys_eq)

print('\n=== Spherical Triangle Area Distribution (after tension iteration) ===')
print(f'triangle_count = {areas_eq.size}')
print(f'min   = {areas_eq.min():.10f}')
print(f'max   = {areas_eq.max():.10f}')
print(f'mean  = {areas_eq.mean():.10f}')
print(f'median= {np.median(areas_eq):.10f}')
print(f'std   = {areas_eq.std(ddof=0):.10f}')

plot_optimizer_history_and_distribution(
    hist,
    areas_eq,
    n=N,
    save_path=Path('figures') / 'octant_n_squared_division_with_area_optimizer_history.svg',
)

result_path = Path('results') / f'division_result_{N}_iter1000.json'
save_division_result(result_path, N, positions_eq)


In [ ]:
# Before/after spherical triangle visualization
positions_before = build_projected_positions(N)

plot_before_after_mesh_comparison(
    triangle_keys=triangle_keys_eq,
    positions_before=positions_before,
    positions_after=positions_eq,
    n=N,
    save_path=Path('figures') / 'octant_n_squared_division_with_area_optimizer.svg',
)

In [ ]:
# Coordinate points (text) + permutation error check
point_keys, point_index = build_point_index(positions_eq)

print('\n=== Points AFTER optimization (index: lattice_index -> [x, y, z]) ===')
for key in point_keys:
    idx = point_index[key]
    i, j = key
    x, y, z = positions_eq[i, j]
    print(f'{idx:3d}: {key} -> [{x:.8f}, {y:.8f}, {z:.8f}]')

print(f'\npoint_count = {len(point_keys)}')

max_perm_err, worst_case = positions_permutation_error(positions_eq)
print('\n=== Permutation Equivariance (optimized positions) ===')
print(f'max_permutation_error = {max_perm_err:.3e}')
print('worst_case =', worst_case)